# Fundamentals 6 · Backpropagation Concepts, Then PyTorch

Notes on chapters 3 and 4 of 3Blue1Brown's *Deep Learning* series, followed by a PyTorch regressor that does
automatically what Fundamentals 7 and the rest of this project implement by hand.

- **Chapter 3 — What is backpropagation really doing?** Derivatives as "nudges", and the different kinds of
  gradient descent.
- **Chapter 4 — Backpropagation calculus.** From a chain of three neurons to a multilayer network, and why the
  chain rule turns into sums.

## Backpropagation concepts

### Key ideas

* Present the idea of gradient as sensitivity.  
* Weights connected to brighter neurons have more influence and are more important to change towards the target.  
* The drunk man analogy that was used to explain stochastic gradient  
* Mini-batches make training more practical.
* ∂w as a tiny nudge to the weight, such as a change of 0.01, and ∂C as the resulting nudge to the cost.
* The effect of a weight nudge depends on how strong the previous neuron is.

**Why use mini-batches instead of the entire dataset for each gradient descent step?**

Adding up the influence of every training example for every step takes a computer an extremely long time.
Randomly dividing the data into mini-batches gives a good approximation of the gradient and a significant
speedup.

### Kinds of gradient descent

Chapter 3 starts from **full-batch** gradient descent (around 9:00), then introduces **stochastic** gradient
descent, and ends with **mini-batch** gradient descent — the variant `numpy_trainer` uses.

### Several paths: the chain rule becomes a sum

When a neuron in previous layer $L_{i-1}$ connects to several neurons in layer $L_{i}$, a change to its activation is passed through every neurons affecting the cost. In the two-path example, $a_k^{L-1}$ affects the cost through $a_0^{L}$ and through $a_1^{L}$, and each path gives its own chain rule product of three ratios. Since the cost only feels the combined effect, you add the products together to get the total sensitivity, with one term for every path.

### "Neurons that fire together wire together"

"Neurons that fire together wire together" means that the connection between two neurons gets stronger when
both are active at the same time. In the gradient below this is the $a^{L-1}$ factor: a weight changes the
most when the neuron feeding into it is strongly active.

$\frac{∂C_o}{∂w^L} = \frac{∂C_o}{∂a^L} * \frac{∂a^L}{∂z^L} * \frac{∂z^L}{∂w^L}$

$\frac{∂C_o}{∂w^L} = a^{L-1} · σ'^{z^L} · 2(a^{L} − y)$

## A PyTorch regressor for concrete strength

A two-layer network (8 → 32 → 1, ReLU) predicts the compressive strength of concrete from the other 8
columns of `data/concrete/train.csv`. PyTorch's autograd computes every gradient that Fundamentals 7 computes
by hand.

In [41]:
from pathlib import Path

# the repository root is the first folder above this notebook that holds pyproject.toml
REPO_ROOT = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / "pyproject.toml").exists())
DATA_DIR = REPO_ROOT / "data"

import torch
import pandas as pd
import numpy as np

from torch import nn
from torch.utils.data import Dataset, DataLoader

### Data

In [42]:
df = pd.read_csv(DATA_DIR / "concrete" / "train.csv")

features = df.iloc[:, :-1]
targets = df.iloc[:, [-1]]

print(f"Features: {features.size} | {features.shape}")
print(f"Targets: {targets.size} | {targets.shape}")

display(features.head(1))
display(targets.head(1))

class ConcreteDataset(Dataset):
    def __init__(self, features, targets):
        self.np_x_set = features.to_numpy()
        self.np_y_set = targets.to_numpy()

        self.np_x_set = self._normalize(self.np_x_set)

        self.x_set = torch.tensor(self.np_x_set, dtype=torch.float32)
        self.y_set = torch.tensor(self.np_y_set, dtype=torch.float32)

    def _normalize(self, a_set):
        return (a_set - a_set.mean(axis=0)) / a_set.std(axis=0)

    def __len__(self):
        return len(self.y_set)

    def __getitem__(self, idx):
        return self.x_set[idx], self.y_set[idx]

dataset = ConcreteDataset(features, targets)
data_loader = DataLoader(dataset, batch_size=64, shuffle=True)

Features: 5600 | (700, 8)
Targets: 700 | (700, 1)


,Cement (component 1)(kg in a m^3 mixture),Blast Furnace Slag (component 2)(kg in a m^3 mixture),Fly Ash (component 3)(kg in a m^3 mixture),Water (component 4)(kg in a m^3 mixture),Superplasticizer (component 5)(kg in a m^3 mixture),Coarse Aggregate (component 6)(kg in a m^3 mixture),Fine Aggregate (component 7)(kg in a m^3 mixture),Age (day)
0,233.81,0.0,94.58,197.89,4.567,947.04,852.16,28


,"Concrete compressive strength(MPa, megapascals)"
0,22.835445


### Model

In [43]:
X_IN = 8
HIDDEN_SIZE = 32
Y_OUT = 1

class Regressor(nn.Module):
    def __init__(self, in_size=X_IN, n_hidden=HIDDEN_SIZE, out_size =Y_OUT):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(in_size, n_hidden),
            nn.ReLU(),
            nn.Linear(n_hidden, out_size)
        )

    def forward(self, x):
        return self.layers(x)

### Training

In [ ]:
EPOCHS = 5000
LEARNING_RATE = 0.003

model = Regressor()

opt = torch.optim.SGD(
    params=model.parameters(),
    lr=LEARNING_RATE
)

cost = nn.MSELoss()
loss = torch.tensor(0.0)

for epoch in range(EPOCHS):
    for x_batch, y_batch in data_loader:
        opt.zero_grad()
        loss = cost(model(x_batch), y_batch)
        loss.backward()
        opt.step()

    if (epoch + 1) % 500 == 0:
        print(f"Epoch {epoch + 1:4d} | Batch Loss: {loss.item():.2f}")

# Final Error
with torch.no_grad():
    train_mse = cost(model(dataset.x_set), dataset.y_set).item()

print(f"Training MSE: {train_mse:.2f}")

Epoch  500 | Batch Loss: 28.70
Epoch 1000 | Batch Loss: 16.22
Epoch 1500 | Batch Loss: 19.91
Epoch 2000 | Batch Loss: 17.91
Epoch 2500 | Batch Loss: 18.18
Epoch 3000 | Batch Loss: 20.87
Epoch 3500 | Batch Loss: 15.15
Epoch 4000 | Batch Loss: 12.44
Epoch 4500 | Batch Loss: 15.03
Epoch 5000 | Batch Loss: 21.10
Training MSE: 14.42


### Against the baseline

The network's MSE compared with the simple bias regressor from Fundamentals 3:

In [45]:
bias = dataset.y_set.mean()
bias_pred = bias.expand_as(dataset.y_set)

with torch.no_grad():
    bias_mse = cost(bias_pred, dataset.y_set).item()

print(f"Bias (mean) prediction: {bias.item():.2f}")
print(f"Bias regressor MSE:     {bias_mse:.2f}")
print(f"Neural network MSE:     {train_mse:.2f}")

Bias (mean) prediction: 36.19
Bias regressor MSE:     275.61
Neural network MSE:     14.42
